In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# Explicit path: the kernel's working directory may not be this folder.
# override=True replaces any empty GROQ_API_KEY already set in this kernel.
load_dotenv(Path("/Users/jobayerahmed/Documents/ML/langchain/ollama/.env"), override=True)
assert os.getenv("GROQ_API_KEY"), "GROQ_API_KEY not found - check .env"

model = ChatGroq(model_name="openai/gpt-oss-120b")

In [3]:
from langchain_core.messages import HumanMessage

In [4]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are helpful assistant. Answer all the question to the best of your ability'),
     MessagesPlaceholder(variable_name="message")
])
chain = prompt|model

In [5]:
chain.invoke({"message": [HumanMessage(content="My name is jobayer ahmed")]})

AIMessage(content='Nice to meet you, Jobayer Ahmed! How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "My name is jobayer ahmed". Likely they want acknowledgement or response. We can respond politely. Probably ask if they need assistance.'}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 96, 'total_tokens': 154, 'completion_time': 0.124438152, 'completion_tokens_details': {'reasoning_tokens': 33}, 'prompt_time': 0.014422189, 'prompt_tokens_details': None, 'queue_time': 0.322740886, 'total_time': 0.138860341}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c800245357', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a169af-20cb-7db3-8263-71ac1c35b14f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 58, 'total_tokens': 154, 'output_token_details': {'reasoning': 33}})

## Message history

In [6]:
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [7]:
# One chat history per session_id (separate name so `chain` isn't overwritten)
store = {}

def get_message_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [8]:
# Wrap the chain: past messages of the session get added in front of "message"
with_message_history = RunnableWithMessageHistory(
    chain,
    get_message_history,
    input_messages_key="message",
)
config = {"configurable": {"session_id": "chat1"}}

### With history

In [9]:
response = with_message_history.invoke(
    {"message": [HumanMessage(content="Hi, my name is Jobayer Ahmed")]},
    config=config,
)
response.content

/var/folders/d0/tvnvw1q168902k33x22mkq340000gn/T/ipykernel_17612/600710899.py:6: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  store[session_id] = InMemoryChatMessageHistory()


'Hello, Jobayer Ahmed! Nice to meet you. How can I assist you today?'

In [10]:
# Same session -> the model remembers the name
response = with_message_history.invoke(
    {"message": [HumanMessage(content="What is my name?")]},
    config=config,
)
response.content

'Your name is **Jobayer Ahmed**.'

### Without history

In [11]:
# Plain chain has no memory -> the model doesn't know the name
response = chain.invoke({"message": [HumanMessage(content="What is my name?")]})
response.content

'I’m not able to see your name from our conversation. If you’d like me to address you by name, just let me know what it is!'

In [12]:
# New session_id starts with an empty history -> doesn't know the name either
response = with_message_history.invoke(
    {"message": [HumanMessage(content="What is my name?")]},
    config={"configurable": {"session_id": "chat2"}},
)
response.content

'I’m not able to see your name from our conversation—unless you’ve shared it with me earlier, I don’t have that information. If you’d like, feel free to tell me your name, and I can address you accordingly!'

In [13]:
# Messages stored for chat1
store["chat1"].messages

[HumanMessage(content='Hi, my name is Jobayer Ahmed', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hello, Jobayer Ahmed! Nice to meet you. How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "Hi, my name is Jobayer Ahmed". Likely they want a response greeting them. Should respond politely, maybe ask how can help.'}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 97, 'total_tokens': 157, 'completion_time': 0.125016164, 'completion_tokens_details': {'reasoning_tokens': 33}, 'prompt_time': 0.005057071, 'prompt_tokens_details': None, 'queue_time': 0.397697154, 'total_time': 0.130073235}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e5b4e54fbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a169af-233e-7d41-8efd-b0540c2856e1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 97, 'output_tokens': 60, 'to

# Managing chat history

In [40]:
from langchain_core.messages import SystemMessage, trim_messages

trimmer = trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

In [41]:
from langchain_core.messages import AIMessage

messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage (content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage (content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage (content= "thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content= "having fun?"),
    AIMessage(content="yes!"),
]

trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [1]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
chain = (
    RunnablePassthrough.assign(messages=itemgetter("message")|trimmer)
    | prompt
    | model
)

NameError: name 'trimmer' is not defined

In [44]:
response = chain.invoke(
    {
    "message": messages+[HumanMessage(content="what math i solved?")],
    "language": "English"
    }
)

response.content

'You just solved a basic addition problem:\n\n**2\u202f+\u202f2 = 4**\n\nIt’s a simple arithmetic calculation. If you’d like to try something a bit more challenging—like multiplication, fractions, or algebra—just let me know!'

In [4]:
import random
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [5]:
class Value:
  def __init__(self, data, _children=(), _op='', label='') -> None:
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label

  def __repr__(self) -> str:
      return f"Value(data={self.data})"

  def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')
    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward
    return out

  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')
    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward
    return out

  def __rmul__(self, other):
    return self * other

  def __pow__(self, other):
    assert isinstance(other, (int, float)), "only supporting int/float powers for now"
    out = Value(self.data**other, (self,), f'**{other}')
    def _backward():
      self.grad += (other * self.data**(other-1)) * out.grad
    out._backward = _backward
    return out

  def tanh(self):
    x = self.data
    t = (math.exp(2*x)-1)/(math.exp(2*x)+1)
    out = Value(t, (self, ), 'tanh')
    def _backward():
      self.grad += (1 - t**2) * out.grad
    out._backward = _backward
    return out
  
  def exp(self):
    x = self.data
    out = Value(math.exp(x), (self, ), 'exp')
    def _backward():
      self.grad += out.data * out.grad
    out._backward = _backward
    return out
  
  def __truediv__(self, other):
    return self * other**-1

  def __neg__(self):
    return self * -1
  
  def __sub__(self, other):
    return self + (-other)

  def backward(self):
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)

    self.grad = 1.0
    for node in reversed(topo):
      node._backward()



In [ ]:
class Neuron:
  def __init__(self, nin):
    self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
    self.b = Value(random.uniform(-1,1))

  def __call__(self, x):
    act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
    out = act.tanh()
    return out

  def parameters(self):
    return self.w + [self.b]



Value(data=-0.9998385708635619)